# 128-bits of security

In [6]:
le_path = "/home/sidsabh/code/crypto/lattice-estimator"
import subprocess
import sys
sys.path.append(le_path)

In [7]:
def checkout(version):
    subprocess.run(["git", "fetch", "--all"], cwd=le_path, capture_output=True)

    if version == "stable":
        subprocess.run(["git", "checkout", "main"], cwd=le_path, capture_output=True)

    elif version == "YPIR":
        subprocess.run(["git", "checkout", "4195c66"], cwd=le_path, capture_output=True)

    elif version == "InsPIRing":
        commit = subprocess.check_output(
            ["git", "rev-list", "-1", "--before=2025-07-24", "main"],
            cwd=le_path
        ).decode().strip()

        subprocess.run(["git", "checkout", commit], cwd=le_path, capture_output=True)

In [8]:
cf = math.sqrt(2*math.pi)
class LatticeParams:
    def __init__(self, n, q, Xs, Xe, m, tag):
        self.n = n
        self.q = q
        self.Xs = Xs
        self.Xe = Xe
        self.m = m
        self.tag = tag

    def estimate(self):
        return LWE.estimate(LWEParameters(n=self.n, q=self.q, Xs=self.Xs, Xe=self.Xe, m=self.m, tag=self.tag))
    
    def estimate_rough(self):
        return LWE.estimate.rough(LWEParameters(n=self.n, q=self.q, Xs=self.Xs, Xe=self.Xe, m=self.m, tag=self.tag))


In [ ]:
import importlib
import estimator

for version in ["stable", "YPIR", "InsPIRing"]:
    checkout(version)

    # force full reimport of estimator after checkout changes the source
    for mod_name in list(sys.modules.keys()):
        if mod_name.startswith("estimator"):
            del sys.modules[mod_name]
    importlib.invalidate_caches()
    import estimator
    from estimator import LWE
    from estimator.lwe_parameters import LWEParameters
    from estimator.nd import NoiseDistribution as ND
    if hasattr(ND, "DiscreteGaussian"):
        D = ND.DiscreteGaussian
    else:
        from estimator.nd import DiscreteGaussian as D


    print(f"\n{'='*60}")
    print(f"version: {version}")
    print(f"{'='*60}")

    print("First Layer")
    LatticeParams(n=1024, q=2**32, Xs=D(11), Xe=D(11), m=2**18, tag='YPIR').estimate()

    print("Second Layer")
    LatticeParams(n=2048, q=268369921*249561089, Xs=D(6.4), Xe=D(6.4), m=2**18, tag='YPIR-SP').estimate()


version: stable
First Layer
bkw                  :: rop: ≈2^338.7, m: ≈2^324.5, mem: ≈2^325.5, b: 10, t1: 0, t2: 44, ℓ: 9, #cod: 974, #top: 0, #test: 51, tag: coded-bkw
usvp                 :: rop: ≈2^130.1, red: ≈2^130.1, δ: 1.004356, β: 351, d: 2192, tag: usvp
bdd                  :: rop: ≈2^128.4, red: ≈2^128.0, svp: ≈2^126.5, β: 343, η: 376, d: 2248, tag: bdd
dual                 :: rop: ≈2^133.0, mem: ≈2^85.1, m: 1276, β: 357, d: 2300, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^134.0, red: ≈2^134.0, guess: ≈2^94.3, β: 362, p: 3, ζ: 0, t: 0, β': 372, N: ≈2^77.1, m: 1024
Second Layer
bkw                  :: rop: ≈2^524.1, m: ≈2^508.2, mem: ≈2^509.2, b: 9, t1: 0, t2: 70, ℓ: 8, #cod: 1963, #top: 1, #test: 84, tag: coded-bkw
usvp                 :: rop: ≈2^132.6, red: ≈2^132.6, δ: 1.004315, β: 356, d: 4108, tag: usvp
bdd                  :: rop: ≈2^131.5, red: ≈2^131.2, svp: ≈2^129.0, β: 351, η: 385, d: 4216, tag: bdd
dual                 :: rop: ≈2^134.5, mem: ≈2^86.2, m: ≈2^11.

In [232]:
import re
import math

sigma = 6.4
while True:
    params = LatticeParams(n=2048, q=2**64, Xs=D(6.4), Xe=D(sigma), m=2**18, tag='Word-SP')
    est = params.estimate_rough()
    print(est)
    min_rop_log2 = min(
        math.log2(float(v['rop']))
        for v in est.values()
    )
    if min_rop_log2 >= 104:
        print(f"Achieved {min_rop_log2:.1f}-bit security at sigma={sigma:.1f}")
        break
    else:
        print(f"rop=2^{min_rop_log2:.1f}, increasing sigma...")
        sigma += 100

usvp                 :: rop: ≈2^84.1, red: ≈2^84.1, δ: 1.004975, β: 288, d: 4080, tag: usvp
dual_hybrid          :: rop: ≈2^85.0, red: ≈2^85.0, guess: ≈2^71.3, β: 291, p: 3, ζ: 0, t: 0, β': 291, N: ≈2^55.6, m: ≈2^11.0
{'usvp': rop: ≈2^84.1, red: ≈2^84.1, δ: 1.004975, β: 288, d: 4080, tag: usvp, 'dual_hybrid': rop: ≈2^85.0, red: ≈2^85.0, guess: ≈2^71.3, β: 291, p: 3, ζ: 0, t: 0, β': 291, N: ≈2^55.6, m: ≈2^11.0}
rop=2^84.1, increasing sigma...
usvp                 :: rop: ≈2^93.1, red: ≈2^93.1, δ: 1.004648, β: 319, d: 4126, tag: usvp
dual_hybrid          :: rop: ≈2^94.0, red: ≈2^94.0, guess: ≈2^82.4, β: 322, p: 3, ζ: 0, t: 0, β': 322, N: ≈2^66.7, m: ≈2^11.0
{'usvp': rop: ≈2^93.1, red: ≈2^93.1, δ: 1.004648, β: 319, d: 4126, tag: usvp, 'dual_hybrid': rop: ≈2^94.0, red: ≈2^94.0, guess: ≈2^82.4, β: 322, p: 3, ζ: 0, t: 0, β': 322, N: ≈2^66.7, m: ≈2^11.0}
rop=2^93.1, increasing sigma...
usvp                 :: rop: ≈2^95.5, red: ≈2^95.5, δ: 1.004571, β: 327, d: 4144, tag: usvp
dual_hybrid     

In [106]:
print(sigma)

138.4


In [214]:
LatticeParams(n=2048, q=268369921 * 249561089, Xs=D(6.4), Xe=D(6.4), m=2**18, tag='Word-SP').estimate_rough()

usvp                 :: rop: ≈2^104.0, red: ≈2^104.0, δ: 1.004315, β: 356, d: 4108, tag: usvp
dual_hybrid          :: rop: ≈2^105.1, red: ≈2^105.1, guess: ≈2^86.9, β: 360, p: 3, ζ: 0, t: 0, β': 360, N: ≈2^71.2, m: ≈2^11.0


{'usvp': rop: ≈2^104.0, red: ≈2^104.0, δ: 1.004315, β: 356, d: 4108, tag: usvp,
 'dual_hybrid': rop: ≈2^105.1, red: ≈2^105.1, guess: ≈2^86.9, β: 360, p: 3, ζ: 0, t: 0, β': 360, N: ≈2^71.2, m: ≈2^11.0}

In [230]:
LatticeParams(n=2048, q=2**64, Xs=D(6.4), Xe=D(1762), m=2**18, tag='Word-SP').estimate()

bkw                  :: rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw
usvp                 :: rop: ≈2^132.6, red: ≈2^132.6, δ: 1.004315, β: 356, d: 4107, tag: usvp
bdd                  :: rop: ≈2^132.0, red: ≈2^132.0, svp: ≈2^126.0, β: 354, η: 375, d: 4066, tag: bdd
dual                 :: rop: ≈2^134.5, mem: ≈2^86.2, m: ≈2^11.1, β: 359, d: 4306, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^134.5, red: ≈2^134.5, guess: ≈2^91.8, β: 359, p: 3, ζ: 0, t: 0, β': 374, N: ≈2^76.1, m: ≈2^11.0


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'bkw': rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw,
 'usvp': rop: ≈2^132.6, red: ≈2^132.6, δ: 1.004315, β: 356, d: 4107, tag: usvp,
 'bdd': rop: ≈2^132.0, red: ≈2^132.0, svp: ≈2^126.0, β: 354, η: 375, d: 4066, tag: bdd,
 'bdd_hybrid': rop: ≈2^132.3, red: ≈2^132.2, svp: ≈2^128.2, β: 354, η: 383, ζ: 0, |S|: 1, d: 4586, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^258.5, red: ≈2^258.5, svp: ≈2^149.9, β: 357, η: 2, ζ: 0, |S|: 1, d: 4600, prob: ≈2^-123.3, ↻: ≈2^125.5, tag: hybrid,
 'dual': rop: ≈2^134.5, mem: ≈2^86.2, m: ≈2^11.1, β: 359, d: 4306, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^134.5, red: ≈2^134.5, guess: ≈2^91.8, β: 359, p: 3, ζ: 0, t: 0, β': 374, N: ≈2^76.1, m: ≈2^11.0}

In [234]:
LatticeParams(n=2048, q=2**64, Xs=D(6.4), Xe=D(6.4*256), m=2**18, tag='Word-SP').estimate()

bkw                  :: rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw
usvp                 :: rop: ≈2^132.1, red: ≈2^132.1, δ: 1.004331, β: 354, d: 4259, tag: usvp
bdd                  :: rop: ≈2^131.7, red: ≈2^131.7, svp: ≈2^125.7, β: 353, η: 374, d: 4064, tag: bdd
dual                 :: rop: ≈2^134.2, mem: ≈2^85.9, m: ≈2^11.1, β: 358, d: 4306, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^134.2, red: ≈2^134.2, guess: ≈2^91.3, β: 358, p: 3, ζ: 0, t: 0, β': 373, N: ≈2^75.6, m: ≈2^11.0


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'bkw': rop: ≈2^532.6, m: ≈2^516.9, mem: ≈2^517.9, b: 8, t1: 0, t2: 61, ℓ: 7, #cod: 1958, #top: 0, #test: 92, tag: coded-bkw,
 'usvp': rop: ≈2^132.1, red: ≈2^132.1, δ: 1.004331, β: 354, d: 4259, tag: usvp,
 'bdd': rop: ≈2^131.7, red: ≈2^131.7, svp: ≈2^125.7, β: 353, η: 374, d: 4064, tag: bdd,
 'bdd_hybrid': rop: ≈2^132.0, red: ≈2^131.9, svp: ≈2^127.9, β: 353, η: 382, ζ: 0, |S|: 1, d: 4582, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^251.3, red: ≈2^251.3, svp: ≈2^142.6, β: 357, η: 2, ζ: 0, |S|: 1, d: 4600, prob: ≈2^-116.1, ↻: ≈2^118.3, tag: hybrid,
 'dual': rop: ≈2^134.2, mem: ≈2^85.9, m: ≈2^11.1, β: 358, d: 4306, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^134.2, red: ≈2^134.2, guess: ≈2^91.3, β: 358, p: 3, ζ: 0, t: 0, β': 373, N: ≈2^75.6, m: ≈2^11.0}

# correctness

In [276]:
import math

W = 1 << 64   # word modulus Z_{2^64}
Q = 268369921 * 249561089   # RLWE CRT modulus

# ─── Base class with shared logic ────────────────────────────────────────

class _CorrectnessBase:
    d   = 2048                            # poly_len
    s   = 6.4 * math.sqrt(2 * math.pi)   # subgaussian parameter sigma_s
    q2  = Q                               # RLWE modulus
    q21 = 268369921                       # a-part modulus after final modswitch
    q22 = 1 << 20                         # b-part modulus after final modswitch
    z   = 1 << 19                         # gadget base
    t   = math.ceil(math.log(Q, z))  # gadget levels

    def __init__(self, l1, l2, packing='inspiring', **kw):
        self.l1, self.l2, self.packing = l1, l2, packing
        for k, v in kw.items():
            setattr(self, k, v)

    @property
    def rho(self):  return math.ceil(self.l2 / self.d)
    @property
    def pt_bits(self):  return int(math.log2(self.p))

    def sigma_pack_sq(self):
        d, s, z, t = self.d, self.s, self.z, self.t
        if   self.packing == 'cdks':       return (d**2 - 1) * t * d * z**2 * s**2 / (4 * 3)
        elif self.packing == 'inspiring':  return t * d**2 * z**2 * s**2 / 4
        else:                              return 0   # hintless

    def sigma_sq(self):  return sum(self.sigma_sq_breakdown().values())

    def log2_delta(self):
        tau, s2 = self.tau(), self.sigma_sq()
        return math.log2(2 * self.d * self.rho) + (-math.pi * tau**2 / s2) / math.log(2)


# ─── Ring SimplePIR ──────────────────────────────────────────────────────

class SPPackCorrectness(_CorrectnessBase):
    """Ring SimplePIR: one modswitch Q -> (q21, q22).

    Deterministic [mod_switch.md, two-modulus]:
        det_final = (2 + q22%p + (q22/Q)(Q%p)) / 2
    Stochastic:
        ms   = (q22/q21)^2 d s^2/4         [a-rounding x secret]
        scan = (q22/Q)^2 (s^2/4) l1 p^2    [LWE noise x DB]
        pack = (q22/Q)^2 sigma_pack^2       [packing noise]
    """
    p = 1 << 15

    def tau_breakdown(self):
        q22, p, q2 = self.q22, self.p, self.q2
        margin    = (q22 - (q22 % p)) / (2 * p)
        det_final = (2 + (q22 % p) + (q22 / q2) * (q2 % p)) / 2
        return {'margin': margin, 'det_final': det_final}

    def tau(self):
        bd = self.tau_breakdown()
        return bd['margin'] - bd['det_final']

    def sigma_sq_breakdown(self):
        d, s, p, q2, q21, q22, l1 = (
            self.d, self.s, self.p, self.q2, self.q21, self.q22, self.l1)
        return {
            'ms':   (q22/q21)**2 * d * s**2 / 4,
            'scan': (q22/q2)**2 * (s**2/4) * l1 * p**2,
            'pack': (q22/q2)**2 * self.sigma_pack_sq(),
        }


# ─── Word SimplePIR ─────────────────────────────────────────────────────

class WordPIRCorrectness(_CorrectnessBase):
    """Word SimplePIR: modswitch W->Q, then packing, then Q->(q21,q22).

    ModSwitch 1 (W -> Q) [single-modulus, mod_switch.md]:
        Deterministic: |delta_W * mu + eps_b| <= (1 + Q%p)/2
            delta_W = (Q%p)/p  [since p | W]
            eps_b   = b-component rounding, |eps_b| <= 1/2
        Stochastic:
            hint_stoch = N s^2/4   [s^T * eps_a, a-rounding x secret]
            scan noise cancels: (Q/W)*sigma_W = sigma_s

    ModSwitch 2 (Q -> q21, q22) [two-modulus]:
        Deterministic: det_final = (2 + q22%p + (q22/Q)(Q%p)) / 2
        Stochastic:    ms = (q22/q21)^2 d s^2/4
        Scales all pre-existing noise by (q22/Q)^2.
        Scales pre-existing det error by (q22/Q).

    Security: D(6.4 * W/Q) ~ D(1762) verified > 128-bit.
    """
    p     = 1 << 15
    n_lwe = 2048

    def tau_breakdown(self):
        q22, p, q2 = self.q22, self.p, self.q2
        margin    = (q22 - (q22 % p)) / (2 * p)
        det_final = (2 + (q22 % p) + (q22 / q2) * (q2 % p)) / 2
        det_wq    = (q22 / q2) * (1 + (q2 % p)) / 2   # W->Q det, scaled to Z_{q22}
        return {'margin': margin, 'det_final': det_final, 'det_wq': det_wq}

    def tau(self):
        bd = self.tau_breakdown()
        return bd['margin'] - bd['det_final'] - bd['det_wq']

    def sigma_sq_breakdown(self):
        d, s, p, q2, q21, q22, l1, n = (
            self.d, self.s, self.p, self.q2, self.q21, self.q22, self.l1, self.n_lwe)
        return {
            'ms':         (q22/q21)**2 * d * s**2 / 4,
            'scan':       (q22/q2)**2 * (s**2/4) * l1 * p**2,
            'pack':       (q22/q2)**2 * self.sigma_pack_sq(),
            'hint_stoch': (q22/q2)**2 * n * s**2 / 4,
        }


# ─── Helpers ─────────────────────────────────────────────────────────────

PT_BITS = {'cdks': 14, 'inspiring': 16, 'hintless': 16}

def sp_dims(num_items, item_size_bits, packing='inspiring'):
    """DB dimensions from item count and size (mirrors params_for_scenario_simplepir)."""
    d = 2048
    pt_bits = PT_BITS[packing]
    instances = math.ceil(item_size_bits / (d * pt_bits))
    nu_1 = max(0, int(math.ceil(math.log2(max(num_items, 1)))) - 11)
    return 1 << (nu_1 + 11), instances * d, 1 << pt_bits

def fmt_size(nbytes):
    if   nbytes >= 1 << 30: return f"{nbytes / (1 << 30):.1f} GiB"
    elif nbytes >= 1 << 20: return f"{nbytes / (1 << 20):.0f} MiB"
    elif nbytes >= 1 << 10: return f"{nbytes / (1 << 10):.0f} KiB"
    return f"{nbytes} B"

In [275]:
# ─── Final correctness table: SimplePIR (ring) vs WordPIR across DB dims ─────

scenarios = [
    (1 << 14, 1 << 22),     # 8 GiB, few large items
    (1 << 16, 1 << 20),     # 8 GiB
    (1 << 18, 1 << 18),     # 8 GiB
    (1 << 20, 1 << 16),     # 8 GiB, many small items
    (1 << 20, 1 << 20),     # 128 GiB
]

header = f"{'DB':>10s}  {'mode':>5s}  {'packing':>10s}  {'pt':>3s}  {'l1':>8s}  {'l2':>8s}  {'rho':>5s}  {'tau':>6s}  {'log2(d)':>8s}  {'dominant':>10s}"
print(header)
print("-" * len(header))
for num_items, item_bits in scenarios:
    db_bytes = num_items * item_bits // 8
    label = fmt_size(db_bytes)
    for pack in ['cdks', 'inspiring']:
        l1, l2, p = sp_dims(num_items, item_bits, pack)
        ring = SPPackCorrectness(l1=l1, l2=l2, packing=pack, p=p)
        word = WordPIRCorrectness(l1=l1, l2=l2, packing=pack, p=p)
        for mode, c in [("ring", ring), ("word", word)]:
            bd = c.sigma_sq_breakdown()
            dominant = max(bd, key=bd.get)
            print(f"{label:>10s}  {mode:>5s}  {pack:>10s}  {c.pt_bits:>3d}  {l1:>8d}  {l2:>8d}  {c.rho:>5d}  {c.tau():>6.2f}  {c.log2_delta():>8.1f}  {dominant:>10s}")
    print()

# ─── Tau & sigma^2 breakdown for representative scenario ─────────────────
print("\n=== Breakdown: 8 GiB (2^16 items x 2^20 bits), InspiRING ===\n")
l1, l2, p = sp_dims(1 << 16, 1 << 20, 'inspiring')
for label, cls in [("Ring SP", SPPackCorrectness), ("Word SP", WordPIRCorrectness)]:
    obj = cls(l1=l1, l2=l2, packing='inspiring', p=p)
    s2 = obj.sigma_sq()

    tau_bd = obj.tau_breakdown()
    print(f"{label}: tau = {obj.tau():.6f},  log2(delta) = {obj.log2_delta():.1f}")
    for k, v in tau_bd.items():
        print(f"  {k:>12}: {v:>14.8f}")

    print(f"  sigma^2 (log2 = {math.log2(s2):.2f}):")
    for k, v in obj.sigma_sq_breakdown().items():
        pct = v / s2 * 100
        lg = math.log2(v) if v > 0 else float('-inf')
        print(f"  {k:>12}: log2 = {lg:>8.1f}  ({pct:5.1f}%)")
    print()

        DB   mode     packing   pt        l1        l2    rho     tau   log2(d)    dominant
-------------------------------------------------------------------------------------------
   8.0 GiB   ring        cdks   14     16384    301056    147   31.00     -91.8        pack
   8.0 GiB   word        cdks   14     16384    301056    147   31.00     -91.8        pack
   8.0 GiB   ring   inspiring   16     16384    262144    128    7.00     -88.5          ms
   8.0 GiB   word   inspiring   16     16384    262144    128    7.00     -88.5          ms

   8.0 GiB   ring        cdks   14     65536     75776     37   31.00     -93.8        pack
   8.0 GiB   word        cdks   14     65536     75776     37   31.00     -93.8        pack
   8.0 GiB   ring   inspiring   16     65536     65536     32    7.00     -90.5          ms
   8.0 GiB   word   inspiring   16     65536     65536     32    7.00     -90.5          ms

   8.0 GiB   ring        cdks   14    262144     20480     10   31.00     -95.

## WordPIR: W=2^32, d=1024, D(11)

In [280]:
# === WordPIR W=2^32, d=1024, D(11), p=2^8 ===
# Goal: minimize q21, q22 (response size); maximize z (fewer gadget levels)
import math
cf = math.sqrt(2 * math.pi)

def is_prime(n):
    if n < 2: return False
    if n < 4: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i*i <= n:
        if n%i == 0 or n%(i+2) == 0: return False
        i += 6
    return True

W32, d32, sig32 = 1 << 32, 1024, 11 * cf
p, p_b = 256, 8

k = (W32 - 2) // 2048
while not is_prime(2048*k+1): k -= 1
Q = 2048*k+1
print(f"Q = {Q}  (2^{math.log2(Q):.4f})\n")

# Sweep all (q21, q22, z) — find Pareto frontier: max z, min response
THRESHOLD = -50   # log2(δ) for rho=1; add log2(rho) for actual DB

results = []
for q22_b in range(10, 22):
    q22 = 1 << q22_b
    if q22 >= Q: break
    for q21_b in range(q22_b, 32):
        q21 = 1 << q21_b
        if q21 >= Q: break
        for z_b in range(3, 18):
            z = 1 << z_b
            t = max(1, math.ceil(math.log(Q, z)))
            obj = WordPIRCorrectness(
                l1=1<<14, l2=d32, packing='inspiring',
                d=d32, s=sig32, n_lwe=d32,
                q2=Q, q21=q21, q22=q22, z=z, t=t, p=p)
            tau = obj.tau()
            if tau <= 0: continue
            ld = obj.log2_delta()
            if ld < THRESHOLD:
                resp_bits = q21_b + q22_b
                results.append((z_b, resp_bits, q21_b, q22_b, t, tau, ld))

# Pareto frontier: for each z, minimum response bits
by_z = {}
for r in results:
    z_b, resp_bits = r[0], r[1]
    if z_b not in by_z or resp_bits < by_z[z_b][1]:
        by_z[z_b] = r

print(f"Pareto frontier (log2δ < {THRESHOLD}, p=2^8):\n")
print(f"{'z':>5}  {'t':>2}  {'q21':>5}  {'q22':>5}  {'tau':>5}  "
      f"{'log2δ':>7}  {'resp/RLWE':>10}")
print("-" * 52)
for z_b in sorted(by_z, reverse=True):
    _, resp_bits, q21_b, q22_b, t, tau, ld = by_z[z_b]
    resp = d32 * resp_bits // 8
    print(f"2^{z_b:<2}  {t:>2}  2^{q21_b:<2}  2^{q22_b:<2}  {tau:>5.1f}  "
          f"{ld:>7.1f}  {fmt_size(resp):>10}")

Q = 4294957057  (2^32.0000)

Pareto frontier (log2δ < -50, p=2^8):

    z   t    q21    q22    tau    log2δ   resp/RLWE
----------------------------------------------------
2^6    6  2^24  2^16  127.0    -50.4       5 KiB
2^5    7  2^21  2^11    3.0    -62.3       4 KiB
2^4    8  2^22  2^10    1.0    -86.7       4 KiB
2^3   11  2^21  2^10    1.0    -57.7       4 KiB


In [331]:
# === WordPIR correctness: W=2^32, d=1024, D(11), p=2^8 (fixed) ===
import math

Q32   = 4294957057
d32   = 1024
sig32 = 11 * math.sqrt(2 * math.pi)
p     = 1 << 8
pt    = 8

num_items      = 1 << 16
item_size_bits = (1 << 17) * 8   # 524288 bits = 64 KiB/item

instances = math.ceil(item_size_bits / (d32 * pt))
l1  = num_items
l2  = instances * d32
rho = math.ceil(l2 / d32)     # = instances = 64
db_bytes = num_items * item_size_bits // 8

print(f"DB: {num_items:,} items × {item_size_bits:,} bits = {fmt_size(db_bytes)}")
print(f"l1 = {l1},  l2 = {l2},  rho = {rho}\n")

# Compare t=6 vs t=7
for label, z_b in [("t=7 (z=2^5)", 5), ("t=6 (z=2^6)", 6), ("t=5 (z=2^7)", 7), ("t=4 (z=2^8)", 8), ("t=8 (z=2^4)", 4)]:
    z = 1 << z_b
    t = math.ceil(math.log(Q32, z))

    # Find minimum (q22, q21) with log2δ < -40
    best = None
    for q22_b in range(9, 32):
        q22 = 1 << q22_b
        if q22 >= Q32: break
        for q21_b in range(q22_b, 32):
            q21 = 1 << q21_b
            if q21 >= Q32: break
            obj = WordPIRCorrectness(
                l1=l1, l2=l2, packing='inspiring',
                d=d32, s=sig32, n_lwe=d32,
                q2=Q32, q21=q21, q22=q22, z=z, t=t, p=p)
            tau = obj.tau()
            if tau <= 0: continue
            ld = obj.log2_delta()
            if ld < -40:
                resp = rho * d32 * (q21_b + q22_b) // 8
                if best is None or resp < best[0]:
                    best = (resp, q21_b, q22_b, t, tau, ld, obj)
                break

    if best is None:
        print(f"{label}: cannot reach log2(δ) < -40 with p=2^8\n")
        continue

    resp, q21_b, q22_b, t, tau, ld, obj = best
    s2 = obj.sigma_sq()
    bd = obj.sigma_sq_breakdown()
    resp_per = d32 * (q21_b + q22_b) // 8

    print(f"{label}:  q21=2^{q21_b}  q22=2^{q22_b}  tau={tau:.0f}  "
          f"log2δ={ld:.1f}  resp={fmt_size(rho*resp_per)}")
    print(f"  σ²: ms={bd['ms']:.2f}  pack={bd['pack']:.2f}  "
          f"scan={bd['scan']:.2f}  hint={bd['hint_stoch']:.4f}")
    print()

DB: 65,536 items × 1,048,576 bits = 8.0 GiB
l1 = 65536,  l2 = 131072,  rho = 128

t=7 (z=2^5):  q21=2^21  q22=2^11  tau=3  log2δ=-40.6  resp=512 KiB
  σ²: ms=0.19  pack=0.32  scan=0.19  hint=0.0000

t=6 (z=2^6): cannot reach log2(δ) < -40 with p=2^8

t=5 (z=2^7): cannot reach log2(δ) < -40 with p=2^8

t=4 (z=2^8): cannot reach log2(δ) < -40 with p=2^8

t=8 (z=2^4):  q21=2^21  q22=2^11  tau=3  log2δ=-69.9  resp=512 KiB
  σ²: ms=0.19  pack=0.09  scan=0.19  hint=0.0000



In [352]:
LatticeParams(n=1024, q=4294957057, Xs=D(1), Xe=D(28), m=2**18, tag='YPIR').estimate()

bkw                  :: rop: ≈2^242.3, m: ≈2^228.2, mem: ≈2^229.2, b: 7, t1: 0, t2: 38, ℓ: 6, #cod: 941, #top: 0, #test: 85, tag: coded-bkw
usvp                 :: rop: ≈2^121.4, red: ≈2^121.4, δ: 1.004638, β: 320, d: 1974, tag: usvp
bdd                  :: rop: ≈2^120.2, red: ≈2^120.0, svp: ≈2^117.4, β: 315, η: 344, d: 1910, tag: bdd
dual                 :: rop: ≈2^123.9, mem: ≈2^78.5, m: 1030, β: 325, d: 2054, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^120.0, red: ≈2^120.0, guess: ≈2^111.5, β: 311, p: 5, ζ: 10, t: 30, β': 325, N: ≈2^67.2, m: 1024


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'bkw': rop: ≈2^242.3, m: ≈2^228.2, mem: ≈2^229.2, b: 7, t1: 0, t2: 38, ℓ: 6, #cod: 941, #top: 0, #test: 85, tag: coded-bkw,
 'usvp': rop: ≈2^121.4, red: ≈2^121.4, δ: 1.004638, β: 320, d: 1974, tag: usvp,
 'bdd': rop: ≈2^120.2, red: ≈2^120.0, svp: ≈2^117.4, β: 315, η: 344, d: 1910, tag: bdd,
 'bdd_hybrid': rop: ≈2^120.7, red: ≈2^120.2, svp: ≈2^119.0, β: 315, η: 350, ζ: 0, |S|: 1, d: 2205, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^228.0, red: ≈2^228.0, svp: ≈2^127.8, β: 323, η: 2, ζ: 0, |S|: 1, d: 2224, prob: ≈2^-103.3, ↻: ≈2^105.5, tag: hybrid,
 'dual': rop: ≈2^123.9, mem: ≈2^78.5, m: 1030, β: 325, d: 2054, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^120.0, red: ≈2^120.0, guess: ≈2^111.5, β: 311, p: 5, ζ: 10, t: 30, β': 325, N: ≈2^67.2, m: 1024}

In [358]:
LatticeParams(n=2048, q=4294957057, Xs=D(1), Xe=D(23), m=2**18, tag='YPIR').estimate()

bkw                  :: rop: ≈2^435.9, m: ≈2^420.5, mem: ≈2^421.5, b: 13, t1: 0, t2: 46, ℓ: 12, #cod: 1889, #top: 0, #test: 165, tag: coded-bkw
usvp                 :: rop: ≈2^251.8, red: ≈2^251.8, δ: 1.002451, β: 785, d: 3892, tag: usvp
bdd                  :: rop: ≈2^251.0, red: ≈2^250.9, svp: ≈2^247.0, β: 782, η: 809, d: 3735, tag: bdd
dual                 :: rop: ≈2^258.0, mem: ≈2^173.8, m: 1978, β: 803, d: 4026, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^245.6, red: ≈2^245.3, guess: ≈2^243.2, β: 758, p: 4, ζ: 10, t: 100, β': 748, N: ≈2^154.8, m: ≈2^11.0


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'bkw': rop: ≈2^435.9, m: ≈2^420.5, mem: ≈2^421.5, b: 13, t1: 0, t2: 46, ℓ: 12, #cod: 1889, #top: 0, #test: 165, tag: coded-bkw,
 'usvp': rop: ≈2^251.8, red: ≈2^251.8, δ: 1.002451, β: 785, d: 3892, tag: usvp,
 'bdd': rop: ≈2^251.0, red: ≈2^250.9, svp: ≈2^247.0, β: 782, η: 809, d: 3735, tag: bdd,
 'bdd_hybrid': rop: ≈2^251.4, red: ≈2^251.2, svp: ≈2^248.6, β: 782, η: 815, ζ: 0, |S|: 1, d: 4303, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^610.1, red: ≈2^610.1, svp: ≈2^380.6, β: 791, η: 2, ζ: 0, |S|: 1, d: 4322, prob: ≈2^-354.2, ↻: ≈2^356.4, tag: hybrid,
 'dual': rop: ≈2^258.0, mem: ≈2^173.8, m: 1978, β: 803, d: 4026, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^245.6, red: ≈2^245.3, guess: ≈2^243.2, β: 758, p: 4, ζ: 10, t: 100, β': 748, N: ≈2^154.8, m: ≈2^11.0}

In [362]:
LatticeParams(n=2048, q=4294957057, Xs=D(1), Xe=D(1), m=2**18, tag='YPIR').estimate_rough()

usvp                 :: rop: ≈2^190.4, red: ≈2^190.4, δ: 1.002810, β: 652, d: 3905, tag: usvp
dual_hybrid          :: rop: ≈2^185.9, red: ≈2^185.7, guess: ≈2^182.7, β: 636, p: 4, ζ: 10, t: 70, β': 636, N: ≈2^131.5, m: ≈2^11.0


{'usvp': rop: ≈2^190.4, red: ≈2^190.4, δ: 1.002810, β: 652, d: 3905, tag: usvp,
 'dual_hybrid': rop: ≈2^185.9, red: ≈2^185.7, guess: ≈2^182.7, β: 636, p: 4, ζ: 10, t: 70, β': 636, N: ≈2^131.5, m: ≈2^11.0}

In [369]:
from estimator.nd import UniformMod as U

In [ ]:
LatticeParams(n=2048, q=4294955009, Xs=U(2), Xe=U(2), m=2**18, tag='YPIR').estimate()

Algorithm <estimator.lwe_bkw.CodedBKW object at 0x738a19845250> on LWEParameters(n=2048, q=4294955009, Xs=D(σ=0.50, μ=-0.50), Xe=D(σ=0.50, μ=-0.50), m=262144, tag='YPIR') failed with Amplifying for μ≠0 not implemented.
arora-gb             :: rop: ≈2^96.2, dreg: 5, mem: ≈2^96.2, t: 0, m: ≈2^18.0, tag: arora-gb, ↻: 1, ζ: 0
usvp                 :: rop: ≈2^201.0, red: ≈2^201.0, δ: 1.002975, β: 603, d: 3780, tag: usvp
bdd                  :: rop: ≈2^199.2, red: ≈2^198.8, svp: ≈2^197.2, β: 595, η: 631, d: 3838, tag: bdd
dual                 :: rop: ≈2^204.8, mem: ≈2^136.0, m: 1886, β: 613, d: 3934, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^192.7, red: ≈2^192.6, guess: ≈2^188.6, β: 569, p: 2, ζ: 0, t: 170, β': 570, N: ≈2^118.1, m: ≈2^11.0


{'arora-gb': rop: ≈2^96.2, dreg: 5, mem: ≈2^96.2, t: 0, m: ≈2^18.0, tag: arora-gb, ↻: 1, ζ: 0,
 'usvp': rop: ≈2^201.0, red: ≈2^201.0, δ: 1.002975, β: 603, d: 3780, tag: usvp,
 'bdd': rop: ≈2^199.2, red: ≈2^198.8, svp: ≈2^197.2, β: 595, η: 631, d: 3838, tag: bdd,
 'bdd_hybrid': rop: ≈2^199.3, red: ≈2^198.9, svp: ≈2^197.2, β: 595, η: 631, ζ: 0, |S|: 1, d: 3893, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^239.4, red: ≈2^238.6, svp: ≈2^238.3, β: 603, η: 2, ζ: 354, |S|: ≈2^354.0, d: 3558, prob: ≈2^-35.5, ↻: ≈2^37.7, tag: hybrid,
 'dual': rop: ≈2^204.8, mem: ≈2^136.0, m: 1886, β: 613, d: 3934, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^192.7, red: ≈2^192.6, guess: ≈2^188.6, β: 569, p: 2, ζ: 0, t: 170, β': 570, N: ≈2^118.1, m: ≈2^11.0}

In [393]:
LatticeParams(n=2048, q=4294955009, Xs=U(2), Xe=D(1), m=2**18, tag='YPIR').estimate()

bkw                  :: rop: ≈2^372.1, m: ≈2^356.6, mem: ≈2^357.6, b: 11, t1: 0, t2: 49, ℓ: 10, #cod: 1848, #top: 0, #test: 202, tag: coded-bkw
usvp                 :: rop: ≈2^207.4, red: ≈2^207.4, δ: 1.002895, β: 626, d: 3769, tag: usvp
bdd                  :: rop: ≈2^205.6, red: ≈2^205.2, svp: ≈2^203.4, β: 618, η: 653, d: 3817, tag: bdd
dual                 :: rop: ≈2^211.6, mem: ≈2^140.9, m: 1879, β: 637, d: 3927, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^198.6, red: ≈2^198.5, guess: ≈2^194.0, β: 590, p: 2, ζ: 10, t: 160, β': 590, N: ≈2^121.3, m: ≈2^11.0


{'arora-gb': rop: ≈2^inf,
 'bkw': rop: ≈2^372.1, m: ≈2^356.6, mem: ≈2^357.6, b: 11, t1: 0, t2: 49, ℓ: 10, #cod: 1848, #top: 0, #test: 202, tag: coded-bkw,
 'usvp': rop: ≈2^207.4, red: ≈2^207.4, δ: 1.002895, β: 626, d: 3769, tag: usvp,
 'bdd': rop: ≈2^205.6, red: ≈2^205.2, svp: ≈2^203.4, β: 618, η: 653, d: 3817, tag: bdd,
 'bdd_hybrid': rop: ≈2^205.6, red: ≈2^205.3, svp: ≈2^203.4, β: 618, η: 653, ζ: 0, |S|: 1, d: 3947, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^247.3, red: ≈2^246.5, svp: ≈2^246.1, β: 625, η: 2, ζ: 366, |S|: ≈2^366.0, d: 3597, prob: ≈2^-37.3, ↻: ≈2^39.5, tag: hybrid,
 'dual': rop: ≈2^211.6, mem: ≈2^140.9, m: 1879, β: 637, d: 3927, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^198.6, red: ≈2^198.5, guess: ≈2^194.0, β: 590, p: 2, ζ: 10, t: 160, β': 590, N: ≈2^121.3, m: ≈2^11.0}

In [392]:
LatticeParams(n=2048, q=4294955009, Xs=U(2), Xe=D(0.5), m=2**18, tag='YPIR').estimate()

bkw                  :: rop: ≈2^366.3, m: ≈2^349.4, mem: ≈2^325.9, b: 10, t1: 4, t2: 55, ℓ: 9, #cod: 1803, #top: 1, #test: 204, tag: coded-bkw
usvp                 :: rop: ≈2^201.0, red: ≈2^201.0, δ: 1.002975, β: 603, d: 3780, tag: usvp
bdd                  :: rop: ≈2^199.2, red: ≈2^198.8, svp: ≈2^197.2, β: 595, η: 631, d: 3838, tag: bdd
dual                 :: rop: ≈2^204.8, mem: ≈2^136.0, m: 1886, β: 613, d: 3934, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^192.7, red: ≈2^192.6, guess: ≈2^188.6, β: 569, p: 2, ζ: 0, t: 170, β': 570, N: ≈2^118.1, m: ≈2^11.0


{'arora-gb': rop: ≈2^inf,
 'bkw': rop: ≈2^366.3, m: ≈2^349.4, mem: ≈2^325.9, b: 10, t1: 4, t2: 55, ℓ: 9, #cod: 1803, #top: 1, #test: 204, tag: coded-bkw,
 'usvp': rop: ≈2^201.0, red: ≈2^201.0, δ: 1.002975, β: 603, d: 3780, tag: usvp,
 'bdd': rop: ≈2^199.2, red: ≈2^198.8, svp: ≈2^197.2, β: 595, η: 631, d: 3838, tag: bdd,
 'bdd_hybrid': rop: ≈2^199.3, red: ≈2^198.9, svp: ≈2^197.2, β: 595, η: 631, ζ: 0, |S|: 1, d: 3893, prob: 1, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^239.4, red: ≈2^238.6, svp: ≈2^238.3, β: 603, η: 2, ζ: 354, |S|: ≈2^354.0, d: 3558, prob: ≈2^-35.5, ↻: ≈2^37.7, tag: hybrid,
 'dual': rop: ≈2^204.8, mem: ≈2^136.0, m: 1886, β: 613, d: 3934, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^192.7, red: ≈2^192.6, guess: ≈2^188.6, β: 569, p: 2, ζ: 0, t: 170, β': 570, N: ≈2^118.1, m: ≈2^11.0}

In [332]:

# === Hybrid RLWE→LWE WordPIR: D(σ_s) / D(σ_e), Q=4294957057, d=1024, p=2^8 ===
#
# Construction:
#   Client: encrypt l1/d RLWE ciphertexts under (Q, D(σ_s), D(σ_e))
#           → modswitch Q→W → reinterpret as l1 scalar LWE in Z_W
#   Server: standard WordPIR matmul in Z_W, modswitch W→Q, pack, modswitch Q→(q21,q22)
#
# Speedup: client does l1/d · d·log(d) mults instead of l1·n → 100× (n=d=1024)
#
# Key change: each query entry has effective noise σ_query = √(σ_e² + d·σ_s²/4)
#   - σ_e²: original RLWE error, preserved through modswitch
#   - d·σ_s²/4: ring product noise s_poly · ε_a from Q→W rounding

import math
cf = math.sqrt(2 * math.pi)

class HybridWordPIRCorrectness(WordPIRCorrectness):
    """Like WordPIRCorrectness but with separate σ_s (secret) and σ_query (query noise).
    
    scan uses σ_query (effective noise per query entry after Q→W modswitch).
    ms, pack, hint use σ_s (the actual secret distribution).
    """
    def __init__(self, sigma_s_sub, sigma_e_sub, **kw):
        self.sigma_s_sub = sigma_s_sub       # subgaussian param of secret
        self.sigma_e_sub = sigma_e_sub       # subgaussian param of RLWE error
        d = kw.get('d', 1024)
        self.sigma_query = math.sqrt(sigma_e_sub**2 + d * sigma_s_sub**2 / 4)
        kw['s'] = sigma_s_sub                # ms/pack/hint use secret param
        super().__init__(**kw)

    def sigma_sq_breakdown(self):
        d, s, p, q2, q21, q22, l1, n = (
            self.d, self.s, self.p, self.q2, self.q21, self.q22, self.l1, self.n_lwe)
        sq = self.sigma_query
        return {
            'ms':         (q22/q21)**2 * d * s**2 / 4,
            'scan':       (q22/q2)**2 * (sq**2/4) * l1 * p**2,   # σ_query, not σ_s
            'pack':       (q22/q2)**2 * self.sigma_pack_sq(),
            'hint_stoch': (q22/q2)**2 * n * s**2 / 4,
        }


# ─── Parameters ──────────────────────────────────────────────────────────
Q32   = 4294957057
d32   = 1024
p     = 256

sigma_s_vals = [2, 1]        # D(σ) secret
sigma_e_vals = [48, 24]      # D(σ) error

print("="*80)
print("Hybrid RLWE→LWE WordPIR: fundamental limits")
print("="*80)
print(f"Q = {Q32} ≈ 2^{math.log2(Q32):.4f},  d = {d32},  p = {p}\n")

# ─── Asymptotic analysis ─────────────────────────────────────────────────
# For large q22 (scan-dominant), τ²/σ² → Q² / (σ_query² · l1 · p⁴)
# Need τ²/σ² > C where log2(δ) = log2(2dρ) - π·C/ln(2) < -40
# → C > (log2(2dρ) + 40) · ln(2)/π

print("─── Asymptotic max l1 (scan-dominant limit) ───\n")
print(f"{'Xs':>5}  {'Xe':>5}  {'σ_s':>7}  {'σ_e':>7}  {'σ_query':>8}  "
      f"{'ms_floor':>8}  {'max_l1':>8}  {'ok?':>5}")
print("-" * 70)

# Target: log2(δ) < -40, with ρ=128 (target DB 2^16 items × 2^17B)
rho_target = 128
C_min = (math.log2(2 * d32 * rho_target) + 40) * math.log(2) / math.pi  # ≈ 12.8

for s_param in [1, 2, 3, 5, 11]:
    ss = s_param * cf
    ms_floor = d32 * ss**2 / 4                   # modswitch noise floor (σ_e=0)
    for e_param in [0, 12, 24, 48, 96]:
        se = e_param * cf
        sq = math.sqrt(se**2 + ms_floor)
        max_l1 = Q32**2 / (C_min * sq**2 * p**4)  # asymptotic
        ok = "YES" if max_l1 >= 65536 else "no"
        if e_param == 0:
            label_e = "min"
        else:
            label_e = f"D({e_param})"
        print(f"D({s_param:>2})  {label_e:>5}  {ss:>7.1f}  {se:>7.1f}  {sq:>8.1f}  "
              f"{ms_floor:>8.0f}  2^{math.log2(max_l1):>4.1f}  {ok:>5}")
    print()

# ─── Detailed check: D(2)/D(48) at target DB ─────────────────────────────
print("\n" + "="*80)
print("Detailed: D(2)/D(48) at target DB (2^16 items × 2^17 B)")
print("="*80)

ss2 = 2 * cf
se48 = 48 * cf
sq_2_48 = math.sqrt(se48**2 + d32 * ss2**2 / 4)

print(f"\nσ_s = {ss2:.2f} (D(2)),  σ_e = {se48:.2f} (D(48))")
print(f"σ_query = √({se48:.1f}² + {d32}·{ss2:.1f}²/4) = √({se48**2:.0f} + {d32*ss2**2/4:.0f}) = {sq_2_48:.1f}")
print(f"  modswitch contributes {d32*ss2**2/4 / sq_2_48**2 * 100:.0f}% of query variance")

# Search best (q22, q21) for l1=2^16
l1 = 65536
l2 = 128 * d32   # 2^17B / (d·pt_bits/8) = 131072/1024 = 128 instances
rho = 128
print(f"\nl1 = {l1},  l2 = {l2},  ρ = {rho}")

best = None
for q22_b in range(9, 28):
    q22 = 1 << q22_b
    if q22 >= Q32: break
    for q21_b in range(q22_b, 32):
        q21 = 1 << q21_b
        if q21 >= Q32: break
        for z_b in [5, 6, 4]:
            z = 1 << z_b
            t = math.ceil(math.log(Q32, z))
            obj = HybridWordPIRCorrectness(
                sigma_s_sub=ss2, sigma_e_sub=se48,
                l1=l1, l2=l2, packing='inspiring',
                d=d32, n_lwe=d32,
                q2=Q32, q21=q21, q22=q22, z=z, t=t, p=p)
            tau = obj.tau()
            if tau <= 0: continue
            ld = obj.log2_delta()
            if ld < -40:
                resp = rho * d32 * (q21_b + q22_b) // 8
                if best is None or resp < best[0]:
                    best = (resp, q21_b, q22_b, t, z_b, tau, ld, obj)

if best is None:
    print("\n*** RESULT: D(2)/D(48) CANNOT achieve log2(δ) < -40 for l1=2^16 ***")
    print("    No (q21, q22, z) combination works.")

    # Show best achievable log2(δ)
    best_ld = float('inf')
    for q22_b in range(9, 28):
        q22 = 1 << q22_b
        if q22 >= Q32: break
        for q21_b in range(q22_b, 32):
            q21 = 1 << q21_b
            if q21 >= Q32: break
            obj = HybridWordPIRCorrectness(
                sigma_s_sub=ss2, sigma_e_sub=se48,
                l1=l1, l2=l2, packing='inspiring',
                d=d32, n_lwe=d32,
                q2=Q32, q21=q21, q22=q22, z=1<<5, t=7, p=p)
            tau = obj.tau()
            if tau <= 0: continue
            ld = obj.log2_delta()
            if ld < best_ld:
                best_ld = ld
                best_params = (q21_b, q22_b, tau, obj)

    q21_b, q22_b, tau, obj = best_params
    bd = obj.sigma_sq_breakdown()
    s2 = sum(bd.values())
    print(f"\n    Best achievable: q21=2^{q21_b}, q22=2^{q22_b}, t={obj.t}")
    print(f"    τ = {tau:.1f},  τ²/σ² = {tau**2/s2:.2f},  log2(δ) = {best_ld:.1f}")
    print(f"    σ² breakdown:")
    for k, v in bd.items():
        lg = math.log2(v) if v > 0 else float('-inf')
        print(f"      {k:>12}: {v:>12.2f}  (log2 = {lg:>6.1f},  {v/s2*100:5.1f}%)")

    # Asymptotic limit
    asym = Q32**2 / (sq_2_48**2 * l1 * p**4)
    print(f"\n    Asymptotic τ²/σ² → {asym:.2f}")
    print(f"    Need τ²/σ² > {C_min:.1f} for log2(δ) < -40")
    print(f"    → Impossible: {asym:.2f} < {C_min:.1f}")
else:
    resp, q21_b, q22_b, t, z_b, tau, ld, obj = best
    print(f"\nBest: q21=2^{q21_b} q22=2^{q22_b} z=2^{z_b} t={t} τ={tau:.0f} log2δ={ld:.1f}")

# ─── Max viable l1 for D(2)/D(48) ────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Max viable l1 for D(2)/D(48), log2(δ) < -40, ρ=1:")
print(f"{'─'*60}")

for target_rho in [1, 8, 64, 128]:
    C_need = (math.log2(2 * d32 * target_rho) + 40) * math.log(2) / math.pi
    max_l1_asym = Q32**2 / (C_need * sq_2_48**2 * p**4)
    print(f"  ρ={target_rho:>3}: max l1 ≈ {int(max_l1_asym):>8,}  (2^{math.log2(max(1,max_l1_asym)):.1f}),  "
          f"need τ²/σ² > {C_need:.1f}")

# ─── Compare: plain D(11) WordPIR (no RLWE conversion) ──────────────────
print(f"\n{'='*80}")
print("Comparison: plain D(11) WordPIR (no RLWE, no modswitch noise)")
print(f"{'='*80}")

s11 = 11 * cf
plain = WordPIRCorrectness(
    l1=65536, l2=128*d32, packing='inspiring',
    d=d32, s=s11, n_lwe=d32,
    q2=Q32, q21=1<<21, q22=1<<11, z=1<<5, t=7, p=p)

asym_plain = Q32**2 / (s11**2 * 65536 * p**4)
print(f"σ_s = {s11:.2f} (D(11) normal-form),  asymptotic τ²/σ² = {asym_plain:.1f}")

best_plain = None
for q22_b in range(9, 22):
    q22 = 1 << q22_b
    for q21_b in range(q22_b, 32):
        q21 = 1 << q21_b
        if q21 >= Q32: break
        obj = WordPIRCorrectness(
            l1=65536, l2=128*d32, packing='inspiring',
            d=d32, s=s11, n_lwe=d32,
            q2=Q32, q21=q21, q22=q22, z=1<<5, t=7, p=p)
        tau = obj.tau()
        if tau <= 0: continue
        ld = obj.log2_delta()
        if ld < -40:
            resp = rho * d32 * (q21_b + q22_b) // 8
            if best_plain is None or resp < best_plain[0]:
                best_plain = (resp, q21_b, q22_b, tau, ld, obj)
            break

if best_plain:
    resp, q21_b, q22_b, tau, ld, obj = best_plain
    bd = obj.sigma_sq_breakdown()
    s2 = sum(bd.values())
    print(f"Best: q21=2^{q21_b} q22=2^{q22_b} τ={tau:.0f} log2δ={ld:.1f} resp={fmt_size(resp)}")
    print(f"  τ²/σ² = {tau**2/s2:.1f}")
    for k, v in bd.items():
        print(f"    {k}: {v:.4f} ({v/s2*100:.1f}%)")

# ─── Root cause ──────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("Root cause:")
print(f"{'='*80}")
print(f"  Plain D(11):   σ_noise = {s11:.1f},    σ² = {s11**2:.0f}")
print(f"  Hybrid D(2)/D(48): σ_query = {sq_2_48:.1f},  σ² = {sq_2_48**2:.0f}")
print(f"  Variance ratio: {sq_2_48**2/s11**2:.1f}×  → max l1 shrinks {sq_2_48**2/s11**2:.1f}×")
print(f"  D(11) max l1 ≈ 2^{math.log2(Q32**2/(C_min*s11**2*p**4)):.1f}")
print(f"  D(2)/D(48) max l1 ≈ 2^{math.log2(Q32**2/(C_min*sq_2_48**2*p**4)):.1f}")
print(f"\n  The modswitch noise floor d·σ_s²/4 = {d32*ss2**2/4:.0f} alone exceeds")
print(f"  the entire D(11) query noise σ²={s11**2:.0f} by {d32*ss2**2/4/s11**2:.1f}×.")
print(f"  Even with σ_e=0, σ_query = √{d32*ss2**2/4:.0f} = {math.sqrt(d32*ss2**2/4):.1f} > {s11:.1f}.")


Hybrid RLWE→LWE WordPIR: fundamental limits
Q = 4294957057 ≈ 2^32.0000,  d = 1024,  p = 256

─── Asymptotic max l1 (scan-dominant limit) ───

   Xs     Xe      σ_s      σ_e   σ_query  ms_floor    max_l1    ok?
----------------------------------------------------------------------
D( 1)    min      2.5      0.0      40.1      1608  2^17.7    YES
D( 1)  D(12)      2.5     30.1      50.1      1608  2^17.0    YES
D( 1)  D(24)      2.5     60.2      72.3      1608  2^16.0     no
D( 1)  D(48)      2.5    120.3     126.8      1608  2^14.3     no
D( 1)  D(96)      2.5    240.6     244.0      1608  2^12.5     no

D( 2)    min      5.0      0.0      80.2      6434  2^15.7     no
D( 2)  D(12)      5.0     30.1      85.7      6434  2^15.5     no
D( 2)  D(24)      5.0     60.2     100.3      6434  2^15.0     no
D( 2)  D(48)      5.0    120.3     144.6      6434  2^14.0     no
D( 2)  D(96)      5.0    240.6     253.7      6434  2^12.3     no

D( 3)    min      7.5      0.0     120.3     14476  2^14.